import copy
# Week 5 — Reinforcement Learning: The Idea

**From Conway to LangGraph: Agent Systems for Physicists in the LLM Era**  
*Department of Physics, University of Bologna — Prof. Degli Esposti*

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Formulate any learning problem in the MDP framework (state, action, reward, policy)
2. Derive the Bellman equation from first principles
3. Implement Q-learning from scratch using pure NumPy
4. Understand ε-annealing as a direct analogue of physical simulated annealing
5. Train an agent on GridWorld and visualise the learned value landscape and policy
6. Verify convergence numerically and connect it to the contraction mapping theorem

## Tools required

```python
import numpy as np
import matplotlib.pyplot as plt
```

No new libraries. Everything runs on what you already have.

---


## 0. Imports and Global Settings

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib import cm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
rng  = np.random.default_rng(SEED)

# Plot style consistent with course theme
plt.rcParams.update({
    'figure.facecolor': '#1A1A1A',
    'axes.facecolor':   '#2D2D2D',
    'axes.edgecolor':   '#555555',
    'axes.labelcolor':  '#AAAAAA',
    'xtick.color':      '#AAAAAA',
    'ytick.color':      '#AAAAAA',
    'text.color':       '#FFFFFF',
    'grid.color':       '#444444',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'lines.linewidth':  2,
    'font.size':        11,
    'figure.dpi':       110,
})

BLUE = '#4A9ECC'
GOLD = '#E8B931'
RED  = '#CC4444'
print("Imports complete. NumPy version:", np.__version__)


---
## 1. The Reinforcement Learning Framework

### From fixed rules to adaptive agents

In Weeks 1–4 every agent followed an immutable rule: Conway's cells, Schelling's households,
sand-pile topplings. Emergence arose from the *structure* of the rules, not from adaptation.

RL changes one thing: the agent can **update its behaviour** based on experience.

### The four ingredients

| Symbol | Name | Meaning |
|--------|------|---------|
| $s_t$ | State | Complete description of the world at time $t$ |
| $a_t$ | Action | Choice made by the agent in state $s_t$ |
| $r_{t+1}$ | Reward | Scalar feedback from the environment |
| $\pi(a \mid s)$ | Policy | Mapping from states to (distributions over) actions |

### The agent–environment loop

At each discrete time step $t$:

$$s_t \xrightarrow{\pi} a_t \xrightarrow{\text{env}} (r_{t+1},\; s_{t+1})$$

The agent's goal: find the policy $\pi^*$ that **maximises total accumulated reward**.

### The Markov property

The environment must satisfy:

$$P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots) = P(s_{t+1} \mid s_t, a_t)$$

The future depends only on the present. This is the physicist's assumption of memorylessness —
the same one underlying Markov chains in statistical mechanics.


### 🔬 Exercise 1.1 — Identify the MDP components

For each scenario below, identify S, A, R, and whether the Markov property holds.

In [ ]:
# This cell is for discussion — no code required.
# Consider the following three systems:

scenarios = {
    "Chess":      {"S": "Board configuration (8x8 positions of all pieces)",
                   "A": "Legal moves (~20 on average, up to 218)",
                   "R": "+1 win, -1 loss, 0 draw",
                   "Markov": "YES — board encodes full game state"},
    "Poker":      {"S": "Visible cards + own hand",
                   "A": "Fold, call, raise",
                   "R": "Chips won/lost at showdown",
                   "Markov": "NO — opponent's cards are hidden (partially observable)"},
    "GridWorld":  {"S": "(row, col) position of agent",
                   "A": "Up, Down, Left, Right",
                   "R": "+1 at goal, -0.01 per step",
                   "Markov": "YES — position fully describes the state"},
}

for name, mdp in scenarios.items():
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for k, v in mdp.items():
        print(f"  {k:10s}: {v}")
    print()


---
## 2. The Physics Perspective

### Free energy and the value function

In statistical mechanics, a system minimises its **Helmholtz free energy**:

$$F = U - TS$$

- $U$ = internal energy (cost to minimise)
- $T$ = temperature (controls randomness / exploration)
- $S$ = entropy (disorder of the state distribution)

In RL, the agent maximises the **value function**:

$$V^\pi(s) = \mathbb{E}_\pi\!\left[\sum_{t=0}^{\infty} \gamma^t r_t \;\middle|\; s_0 = s\right]$$

- $V^\pi(s)$ plays the role of $-F$ (maximise, not minimise)
- $\gamma \in [0,1)$ is the **discount factor** — controls the temporal horizon
- The exploration parameter $\varepsilon$ (introduced later) maps directly onto temperature $T$

The analogy is exact: high $T$ (high $\varepsilon$) → random exploration; low $T$ (low $\varepsilon$) → greedy exploitation.


In [ ]:
# Visualise the role of the discount factor gamma
gammas = [0.5, 0.9, 0.99]
t_max  = 30
t      = np.arange(t_max)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: discount weights over time
ax = axes[0]
for g in gammas:
    ax.plot(t, g**t, label=f'γ = {g}', linewidth=2)
ax.set_xlabel('Time step t')
ax.set_ylabel('Discount weight γᵗ')
ax.set_title('How much future rewards matter', color='white')
ax.legend()
ax.grid(True)

# Right: effective horizon = 1/(1-gamma)
g_vals  = np.linspace(0.01, 0.99, 300)
horizon = 1 / (1 - g_vals)
ax = axes[1]
ax.plot(g_vals, horizon, color=GOLD)
ax.axvline(0.9,  color=BLUE, linestyle='--', alpha=0.7, label='γ=0.90 → horizon=10')
ax.axvline(0.99, color=RED,  linestyle='--', alpha=0.7, label='γ=0.99 → horizon=100')
ax.set_xlabel('Discount factor γ')
ax.set_ylabel('Effective horizon  1/(1−γ)')
ax.set_title('Effective planning horizon', color='white')
ax.legend(fontsize=9)
ax.grid(True)
ax.set_ylim(0, 120)

plt.suptitle('The Discount Factor γ', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("γ=0.9  → effective horizon:", round(1/(1-0.9)))
print("γ=0.99 → effective horizon:", round(1/(1-0.99)))


---
## 3. The Bellman Equation — Derivation

### Step 1: The value function is recursive

Starting from the definition:

$$V^\pi(s) = \mathbb{E}_\pi\!\left[r_0 + \gamma r_1 + \gamma^2 r_2 + \cdots \mid s_0=s\right]$$

Factor out one reward:

$$V^\pi(s) = \mathbb{E}_\pi\!\left[r_0 + \gamma\underbrace{(r_1 + \gamma r_2 + \cdots)}_{V^\pi(s_1)} \;\middle|\; s_0=s\right]$$

$$\boxed{V^\pi(s) = \mathbb{E}_\pi\!\left[r_0 + \gamma\, V^\pi(s_1) \;\middle|\; s_0=s\right]}$$

This is the **self-referential structure** — the value of a state equals immediate reward
plus discounted value of the next state.

### Step 2: Expand the expectation

The agent selects action $a$ with probability $\pi(a \mid s)$.  
The environment transitions to $s'$ with probability $P(s' \mid s, a)$ and emits reward $R(s,a,s')$.

Summing over all actions and next states:

$$\boxed{V^\pi(s) = \sum_a \pi(a \mid s)\, \sum_{s'} P(s' \mid s, a)\!\left[R(s,a,s') + \gamma\, V^\pi(s')\right]}$$

This is the **Bellman Expectation Equation**.

### Step 3: The optimality equation

For the optimal policy $\pi^*$, instead of averaging over $\pi(a|s)$ we take the best action:

$$\boxed{V^*(s) = \max_a \sum_{s'} P(s' \mid s, a)\!\left[R(s,a,s') + \gamma\, V^*(s')\right]}$$

### The fixed-point interpretation

The Bellman operator $\mathcal{T}^\pi$ is a **contraction mapping** with factor $\gamma < 1$:

$$\|\mathcal{T}^\pi V - \mathcal{T}^\pi V'\|_\infty \leq \gamma\, \|V - V'\|_\infty$$

By the Banach fixed-point theorem: **unique fixed point** $V^\pi$, reachable by iteration from any starting point.


### 🔬 Experiment 3.1 — Value iteration converges as a contraction

In [ ]:
# Demonstrate Bellman iteration as a contraction mapping
# Simple 3-state chain: s0 -> s1 -> s2 (absorbing)
# Rewards: r(s0->s1)=0, r(s1->s2)=1
# Deterministic policy, gamma=0.9

gamma = 0.9

# True values (geometric series):  V(s0)=0.9, V(s1)=1.0, V(s2)=0
V_true = np.array([0.9, 1.0, 0.0])

def bellman_step(V, gamma=0.9):
    '''One step of Bellman iteration for the 3-state chain.'''
    V_new = np.zeros(3)
    V_new[0] = 0    + gamma * V[1]   # s0 -> s1, reward=0
    V_new[1] = 1    + gamma * V[2]   # s1 -> s2, reward=1
    V_new[2] = 0                      # absorbing state
    return V_new

# Start from arbitrary initial values
V = np.array([10.0, -5.0, 3.0])
print(f"Initial V:  {V}")
print(f"True    V:  {V_true}")
print()

errors = []
V_history = [V.copy()]

for i in range(30):
    V = bellman_step(V, gamma)
    error = np.max(np.abs(V - V_true))
    errors.append(error)
    V_history.append(V.copy())

print(f"Final   V:  {V}")
print(f"Max error after 30 iterations: {errors[-1]:.2e}")

# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.semilogy(errors, color=BLUE, linewidth=2)
# Overlay theoretical contraction rate gamma^n
n_iter = np.arange(len(errors))
ax.semilogy(n_iter, errors[0] * gamma**n_iter, '--', color=GOLD,
            label=f'Theoretical: γⁿ × ||V₀-V*||∞', linewidth=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Max error  ||V - V*||∞')
ax.set_title('Bellman iteration converges geometrically', color='white')
ax.legend()
ax.grid(True)

ax = axes[1]
V_hist = np.array(V_history)
for i, (label, col) in enumerate(zip(['V(s₀)', 'V(s₁)', 'V(s₂)'], [BLUE, GOLD, RED])):
    ax.plot(V_hist[:, i], color=col, label=label)
    ax.axhline(V_true[i], color=col, linestyle=':', alpha=0.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Value')
ax.set_title('Value estimates converging to V* (dotted)', color='white')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()
print(f"\nContraction rate per step: γ = {gamma}")
print(f"Predicted error after 30 steps: {errors[0] * gamma**30:.2e}")
print(f"Observed  error after 30 steps: {errors[-1]:.2e}")


---
## 4. From V to Q — The Action-Value Function

### Why we need Q instead of V

$V^\pi(s)$ tells the agent how good it is to *be* in state $s$.  
But to act, the agent needs to know which *action* to take.  
Extracting a policy from $V^*$ requires knowing $P(s' \mid s, a)$ — the environment model.

The **Q-function** (action-value function) solves this:

$$Q^\pi(s,a) = \mathbb{E}_\pi\!\left[\sum_{t=0}^{\infty} \gamma^t r_t \;\middle|\; s_0=s,\; a_0=a\right]$$

$Q^\pi(s,a)$ is the expected return when starting in $s$, taking action $a$ first,
then following $\pi$ thereafter.

### Relationship between Q and V

$$V^\pi(s) = \sum_a \pi(a \mid s)\, Q^\pi(s,a)$$

$$Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s' \mid s,a)\, V^\pi(s')$$

### Bellman optimality for Q*

$$\boxed{Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s' \mid s,a)\, \max_{a'} Q^*(s',a')}$$

### Extracting the optimal policy — no model needed

$$\pi^*(s) = \underset{a}{\arg\max}\; Q^*(s,a)$$

Once $Q^*$ is learned, the agent acts optimally by simple lookup — **no knowledge of $P(s'|s,a)$ required**.  
This is what makes Q-learning a **model-free** method.


---
## 5. GridWorld — The Hydrogen Atom of RL

GridWorld is the simplest non-trivial RL environment:

| Property | Value |
|----------|-------|
| Grid size | 5 × 5 |
| State space | 25 cells, each a (row, col) pair |
| Action space | {Up=0, Down=1, Left=2, Right=3} |
| Goal cell | (3, 4) — reward +1, episode ends |
| Wall cells | (1,1), (2,1), (2,3), (3,3) — agent stays put |
| Step reward | −0.01 (encourages finding shortest path) |
| Q-table size | 25 × 4 = 100 entries |

The Q-table is small enough to **print and inspect by hand**.


In [ ]:
class GridWorld:
    """
    5x5 GridWorld environment.

    Grid layout (W=wall, G=goal, S=start):
        . . . . .
        . W . . .
        . W . W .
        . . . W G
        . S . . .

    Actions: 0=Up, 1=Down, 2=Left, 3=Right
    """

    ACTIONS     = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    ACTION_NAME = {0: '↑', 1: '↓', 2: '←', 3: '→'}

    def __init__(self):
        self.rows, self.cols = 5, 5
        self.goal  = (3, 4)
        self.start = (4, 1)
        self.walls = {(1,1), (2,1), (2,3), (3,3)}

        self.R_GOAL = +1.0
        self.R_STEP = -0.01

        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        dr, dc = self.ACTIONS[action]
        r, c   = self.state
        nr, nc = r + dr, c + dc

        # Check bounds and walls
        if 0 <= nr < self.rows and 0 <= nc < self.cols and (nr, nc) not in self.walls:
            self.state = (nr, nc)

        if self.state == self.goal:
            return self.state, self.R_GOAL, True   # (next_state, reward, done)
        else:
            return self.state, self.R_STEP, False

    def render(self, Q=None, ax=None):
        """Visualise the grid, optionally overlaying the greedy policy from Q."""
        show = ax is None
        if ax is None:
            fig, ax = plt.subplots(figsize=(5, 5))

        grid_img = np.zeros((self.rows, self.cols, 3))
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) in self.walls:
                    grid_img[r, c] = [0.24, 0.13, 0.13]   # dark red
                elif (r, c) == self.goal:
                    grid_img[r, c] = [0.10, 0.20, 0.12]   # dark green
                elif (r, c) == self.start:
                    grid_img[r, c] = [0.11, 0.16, 0.24]   # dark blue
                else:
                    grid_img[r, c] = [0.18, 0.18, 0.18]

        ax.imshow(grid_img, interpolation='nearest')

        # Grid lines
        for i in range(self.cols + 1):
            ax.axvline(i - 0.5, color='#555555', linewidth=1)
        for i in range(self.rows + 1):
            ax.axhline(i - 0.5, color='#555555', linewidth=1)

        # Labels
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) in self.walls:
                    ax.text(c, r, '■', ha='center', va='center',
                            color='#CC4444', fontsize=14, fontweight='bold')
                elif (r, c) == self.goal:
                    ax.text(c, r, 'G', ha='center', va='center',
                            color=GOLD, fontsize=14, fontweight='bold')
                elif (r, c) == self.start:
                    ax.text(c, r, 'S', ha='center', va='center',
                            color=BLUE, fontsize=12, fontweight='bold')

        # Policy arrows
        if Q is not None:
            for r in range(self.rows):
                for c in range(self.cols):
                    if (r, c) in self.walls or (r, c) == self.goal:
                        continue
                    best_a = np.argmax(Q[r, c])
                    arrow  = self.ACTION_NAME[best_a]
                    ax.text(c, r, arrow, ha='center', va='center',
                            color='white', fontsize=13, alpha=0.85)

        ax.set_xlim(-0.5, self.cols - 0.5)
        ax.set_ylim(self.rows - 0.5, -0.5)
        ax.set_xticks(range(self.cols))
        ax.set_yticks(range(self.rows))
        ax.set_xticklabels(range(self.cols), color='#666666', fontsize=8)
        ax.set_yticklabels(range(self.rows), color='#666666', fontsize=8)

        if show:
            plt.title('GridWorld', color='white')
            plt.tight_layout()
            plt.show()

# Display the empty environment
env = GridWorld()
fig, ax = plt.subplots(figsize=(4.5, 4.5))
env.render(ax=ax)
ax.set_title('GridWorld: initial layout', color='white', pad=10)
plt.tight_layout()
plt.show()
print("State space  :", env.rows * env.cols, "cells")
print("Wall cells   :", sorted(env.walls))
print("Goal cell    :", env.goal)
print("Start cell   :", env.start)
print("Q-table size :", env.rows, "×", env.cols, "×", len(env.ACTIONS), "=",
      env.rows * env.cols * len(env.ACTIONS), "entries")


---
## 6. Exploration vs. Exploitation — The Temperature Analogy

The ε-greedy policy:

$$a = \begin{cases} \text{random action} & \text{with probability } \varepsilon \\ \arg\max_a Q(s,a) & \text{with probability } 1-\varepsilon \end{cases}$$

### The annealing schedule

$$\varepsilon(t) = \varepsilon_{\min} + (\varepsilon_{\max} - \varepsilon_{\min})\, e^{-t/\tau}$$

| Physical SA | RL ε-annealing |
|-------------|----------------|
| Temperature $T$ | Exploration rate $\varepsilon$ |
| High $T$: atoms explore widely | High $\varepsilon$: agent tries random actions |
| Low $T$: atoms settle in minimum | Low $\varepsilon$: agent exploits learned Q |
| Cooling schedule $T(t)$ | Annealing schedule $\varepsilon(t)$ |
| Too fast → trapped in local minimum | Too fast → suboptimal policy learned |
| Too slow → computationally wasteful | Too slow → excessive random actions |

The cooling timescale $\tau$ is the key hyperparameter — analogous to the cooling rate in metallurgy.


In [ ]:
# Visualise epsilon annealing schedules with different tau values
episodes = np.arange(1000)
eps_min  = 0.01
eps_max  = 1.0
taus     = [100, 300, 700]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for tau in taus:
    eps = eps_min + (eps_max - eps_min) * np.exp(-episodes / tau)
    ax.plot(episodes, eps, label=f'τ = {tau}', linewidth=2)
ax.axhline(eps_min, color='white', linestyle=':', alpha=0.4, label='ε_min')
ax.set_xlabel('Episode')
ax.set_ylabel('ε (exploration rate)')
ax.set_title('ε-Annealing Schedules', color='white')
ax.legend()
ax.grid(True)

# Right: fraction of episodes spent exploring vs exploiting
ax = axes[1]
tau = 300
eps = eps_min + (eps_max - eps_min) * np.exp(-episodes / tau)
ax.fill_between(episodes, eps, 1.0, alpha=0.5, color=BLUE, label='Exploit (1−ε)')
ax.fill_between(episodes, 0, eps, alpha=0.5, color=GOLD, label='Explore (ε)')
ax.plot(episodes, eps, color=GOLD, linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Fraction of actions')
ax.set_title(f'Explore vs Exploit (τ={tau})', color='white')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

# Physical simulated annealing analogy: acceptance probability
print("\n--- Physical SA analogy ---")
print("In SA, a worse solution ΔE > 0 is accepted with probability exp(-ΔE/T)")
print("In RL,  a random action is taken with probability ε")
print()
T_vals = np.array([1.0, 0.5, 0.1, 0.01])
delta_E = 0.5
for T in T_vals:
    p_accept = np.exp(-delta_E / T)
    print(f"  T={T:.2f}  →  P(accept worse) = exp(-0.5/{T:.2f}) = {p_accept:.4f}")


---
## 7. Q-Learning — Full Implementation

### The update rule

After each transition $(s, a, r, s')$:

$$Q(s,a) \;\leftarrow\; Q(s,a) + \alpha \underbrace{\big[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big]}_{\delta_t \;\text{(TD error)}}$$

- $\alpha \in (0,1]$: learning rate
- $\delta_t$: **temporal difference error** — positive means better than expected, negative means worse
- $r + \gamma \max_{a'} Q(s',a')$: TD target — one-step lookahead estimate of $Q^*(s,a)$

### Algorithm

```
Initialise Q(s,a) = 0  ∀ s, a
For each episode:
    s ← reset environment
    While not done:
        a ← ε-greedy(Q, s)
        s', r, done ← env.step(a)
        δ ← r + γ max_{a'} Q(s',a') - Q(s,a)
        Q(s,a) ← Q(s,a) + α·δ
        s ← s'
    Anneal ε
```


In [ ]:
class QLearningAgent:
    """
    Tabular Q-learning agent with ε-greedy exploration and exponential annealing.

    Parameters
    ----------
    n_rows, n_cols : int
        Grid dimensions — define the state space.
    n_actions : int
        Number of discrete actions.
    alpha : float
        Learning rate (Robbins-Monro: 0 < alpha <= 1).
    gamma : float
        Discount factor (0 < gamma < 1).
    eps_max : float
        Initial exploration rate.
    eps_min : float
        Final exploration rate.
    tau : float
        Annealing timescale (in episodes).
    init_value : float
        Initial Q-table value. Use 0 for standard init,
        or a positive number for optimistic initialisation.
    seed : int
        Random seed for reproducibility.
    """

    def __init__(self, n_rows=5, n_cols=5, n_actions=4,
                 alpha=0.1, gamma=0.9,
                 eps_max=1.0, eps_min=0.01, tau=300,
                 init_value=0.0, seed=SEED):

        self.n_rows, self.n_cols = n_rows, n_cols
        self.n_actions = n_actions
        self.alpha     = alpha
        self.gamma     = gamma
        self.eps_max   = eps_max
        self.eps_min   = eps_min
        self.tau       = tau
        self.rng       = np.random.default_rng(seed)

        # Q-table: shape (rows, cols, actions)
        self.Q = np.full((n_rows, n_cols, n_actions), init_value, dtype=float)

        # Tracking
        self.episode = 0
        self.epsilon_history  = []
        self.reward_history   = []
        self.td_error_history = []
        self.steps_history    = []

    @property
    def epsilon(self):
        return self.eps_min + (self.eps_max - self.eps_min) * np.exp(-self.episode / self.tau)

    def select_action(self, state):
        """ε-greedy action selection."""
        if self.rng.random() < self.epsilon:
            return self.rng.integers(self.n_actions)          # explore
        else:
            return np.argmax(self.Q[state[0], state[1]])       # exploit

    def update(self, state, action, reward, next_state, done):
        """One Q-learning update step. Returns the TD error δ."""
        s,  a  = state[0],      state[1]
        s_, a_ = next_state[0], next_state[1]

        q_current = self.Q[s, a, action]

        if done:
            td_target = reward                                # no future after terminal
        else:
            td_target = reward + self.gamma * np.max(self.Q[s_, a_])

        td_error = td_target - q_current
        self.Q[s, a, action] += self.alpha * td_error

        return td_error

    def run_episode(self, env, max_steps=200):
        """Run one complete episode. Returns (total_reward, steps, mean_|δ|)."""
        state = env.reset()
        total_reward = 0.0
        td_errors    = []

        for step in range(max_steps):
            action          = self.select_action(state)
            next_state, r, done = env.step(action)
            delta           = self.update(state, action, r, next_state, done)

            td_errors.append(abs(delta))
            total_reward += r
            state         = next_state

            if done:
                break

        self.episode += 1
        self.epsilon_history.append(self.epsilon)
        self.reward_history.append(total_reward)
        self.td_error_history.append(np.mean(td_errors))
        self.steps_history.append(step + 1)

        return total_reward, step + 1, np.mean(td_errors)


# Quick sanity check: one episode
env   = GridWorld()
agent = QLearningAgent()
r, steps, td = agent.run_episode(env)
print(f"Episode 1: reward={r:.3f}, steps={steps}, mean|δ|={td:.4f}")
print(f"Q-table shape: {agent.Q.shape}")
print(f"Initial ε: {agent.eps_max:.2f}, after episode 1: {agent.epsilon:.4f}")


## 8. Training the Agent

In [ ]:
# Full training run
N_EPISODES = 2000
env        = GridWorld()
agent      = QLearningAgent(alpha=0.1, gamma=0.9, tau=300, eps_max=1.0, eps_min=0.01)

print(f"Training for {N_EPISODES} episodes...")
print(f"{'Episode':>8}  {'Reward':>8}  {'Steps':>6}  {'ε':>6}  {'mean|δ|':>10}")
print("-" * 48)

for ep in range(N_EPISODES):
    r, steps, td = agent.run_episode(env)
    if (ep + 1) % 400 == 0 or ep == 0:
        print(f"{ep+1:>8}  {r:>8.3f}  {steps:>6}  {agent.epsilon:>6.3f}  {td:>10.5f}")

print("\nTraining complete.")
print(f"Final ε = {agent.epsilon:.4f}")


In [ ]:
# Training curves — the physicist's favourite diagnostic
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
window = 50  # smoothing window

def smooth(x, w=window):
    return np.convolve(x, np.ones(w)/w, mode='valid')

episodes_x = np.arange(N_EPISODES)

# 1. Total reward per episode
ax = axes[0, 0]
ax.plot(episodes_x, agent.reward_history, color='#444444', alpha=0.4, linewidth=0.8)
ax.plot(np.arange(len(smooth(agent.reward_history))), smooth(agent.reward_history),
        color=BLUE, linewidth=2, label=f'Moving avg (w={window})')
ax.set_xlabel('Episode')
ax.set_ylabel('Total reward')
ax.set_title('Return per episode', color='white')
ax.legend()
ax.grid(True)

# 2. Steps to reach goal
ax = axes[0, 1]
ax.plot(episodes_x, agent.steps_history, color='#444444', alpha=0.4, linewidth=0.8)
ax.plot(np.arange(len(smooth(agent.steps_history))), smooth(agent.steps_history),
        color=GOLD, linewidth=2, label=f'Moving avg')
ax.set_xlabel('Episode')
ax.set_ylabel('Steps')
ax.set_title('Steps per episode (fewer = better)', color='white')
ax.legend()
ax.grid(True)

# 3. Mean TD error — convergence diagnostic
ax = axes[1, 0]
ax.semilogy(episodes_x, agent.td_error_history, color='#444444', alpha=0.4, linewidth=0.8)
ax.semilogy(np.arange(len(smooth(agent.td_error_history))), smooth(agent.td_error_history),
            color=RED, linewidth=2, label='Moving avg')
ax.set_xlabel('Episode')
ax.set_ylabel('Mean |δ|  (log scale)')
ax.set_title('TD Error — convergence indicator', color='white')
ax.legend()
ax.grid(True)

# 4. Epsilon annealing
ax = axes[1, 1]
ax.plot(episodes_x, agent.epsilon_history, color=GOLD, linewidth=2)
ax.fill_between(episodes_x, 0, agent.epsilon_history, alpha=0.2, color=GOLD)
ax.set_xlabel('Episode')
ax.set_ylabel('ε (exploration rate)')
ax.set_title('ε-Annealing Schedule', color='white')
ax.grid(True)

plt.suptitle('Q-Learning Training Curves — GridWorld 5×5', color='white', fontsize=13)
plt.tight_layout()
plt.show()


## 9. Visualising the Learned Policy and Value Landscape

In [ ]:
# Compute value function from Q-table: V(s) = max_a Q(s,a)
V = np.max(agent.Q, axis=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Panel 1: Learned policy (arrows) ---
env.render(Q=agent.Q, ax=axes[0])
axes[0].set_title('Learned Policy  π*(s)', color='white', pad=10)

# --- Panel 2: Value function heatmap ---
ax = axes[1]
V_display = V.copy()
V_display[1,1] = V_display[2,1] = V_display[2,3] = V_display[3,3] = np.nan  # walls

cmap = plt.cm.plasma
norm = Normalize(vmin=np.nanmin(V_display), vmax=np.nanmax(V_display))
img = ax.imshow(V_display, cmap=cmap, norm=norm, interpolation='nearest')

for r in range(5):
    for c in range(5):
        if (r,c) in env.walls:
            ax.text(c, r, '■', ha='center', va='center', color=RED, fontsize=14)
        elif (r,c) == env.goal:
            goal_label = 'G  ' + f'{V[r,c]:.2f}'
            ax.text(c, r, goal_label, ha='center', va='center',
                    color='white', fontsize=10, fontweight='bold')
        else:
            ax.text(c, r, f'{V[r,c]:.2f}', ha='center', va='center',
                    color='white', fontsize=9)

for i in range(6): ax.axvline(i-0.5, color='#555555', linewidth=0.8)
for i in range(6): ax.axhline(i-0.5, color='#555555', linewidth=0.8)
plt.colorbar(img, ax=ax, label='V*(s)')
ax.set_title('Value Function  V*(s)', color='white', pad=10)
ax.set_xticks(range(5)); ax.set_yticks(range(5))

# --- Panel 3: Q-table heatmap for one slice ---
ax = axes[2]
action_names = ['↑ Up', '↓ Down', '← Left', '→ Right']
# Flatten Q-table: 25 states x 4 actions
Q_flat = agent.Q.reshape(25, 4)

im = ax.imshow(Q_flat, cmap='plasma', aspect='auto', interpolation='nearest')
ax.set_xlabel('Action')
ax.set_ylabel('State index (row*5 + col)')
ax.set_xticks(range(4))
ax.set_xticklabels(action_names, rotation=30, fontsize=9)
ax.set_title('Full Q-Table  Q*(s,a)', color='white', pad=10)
plt.colorbar(im, ax=ax, label='Q value')

plt.suptitle('Learned Policy and Value Landscape — GridWorld', color='white', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Print the full Q-table — small enough to read by hand
print("Full Q-table Q*(s, a)")
print("State (r,c)   |   ↑ Up    ↓ Down   ← Left   → Right  |  Best action")
print("-" * 75)
for r in range(5):
    for c in range(5):
        q = agent.Q[r, c]
        best = np.argmax(q)
        arrow = GridWorld.ACTION_NAME[best]
        tag = ""
        if (r,c) in env.walls: tag = "[WALL]"
        elif (r,c) == env.goal: tag = "[GOAL]"
        elif (r,c) == env.start: tag = "[START]"
        print(f"  ({r},{c})  {tag:7s}  |  {q[0]:7.4f}  {q[1]:7.4f}  {q[2]:7.4f}  {q[3]:7.4f}  |  {arrow}")


---
## 5b. GridWorld — A Complete Walkthrough

This section answers the question: **what exactly is happening, and what does the final answer mean?**

We will go step by step:
1. Understand the grid layout with coordinates
2. Watch a single episode of a *random* (untrained) agent
3. Watch a single episode of the *trained* agent
4. Read the Q-table cell by cell
5. Understand *why* the Q-values have the values they do
6. See how the policy evolved snapshot by snapshot during training


### Step 1 — The Grid Layout: Every Cell Named

Before training anything, let us be completely clear about what the grid looks like.
Every cell has coordinates **(row, col)**, row 0 at the top.


In [ ]:

# ── Paint a fully annotated grid ─────────────────────────────────────────────
env = GridWorld()

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_facecolor('#1A1A1A')
fig.patch.set_facecolor('#1A1A1A')

cell_colors = {
    'empty': '#2D2D2D',
    'wall':  '#3D1A1A',
    'goal':  '#1A3320',
    'start': '#1A2438',
}

for r in range(5):
    for c in range(5):
        pos = (r, c)
        if pos in env.walls:
            fc = cell_colors['wall']
            label = 'WALL'
            tc = '#CC4444'
        elif pos == env.goal:
            fc = cell_colors['goal']
            label = 'GOAL\n+1'
            tc = GOLD
        elif pos == env.start:
            fc = cell_colors['start']
            label = 'START'
            tc = BLUE
        else:
            fc = cell_colors['empty']
            label = ''
            tc = '#888888'

        rect = plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                              facecolor=fc, edgecolor='#555555', linewidth=1.5)
        ax.add_patch(rect)

        # Coordinate label (small, top-left of cell)
        ax.text(c - 0.38, r - 0.38, f'({r},{c})',
                fontsize=7.5, color='#666666', va='top', ha='left')

        # Cell type label (centre)
        if label:
            ax.text(c, r, label, fontsize=11, color=tc,
                    ha='center', va='center', fontweight='bold')

ax.set_xlim(-0.5, 4.5)
ax.set_ylim(4.5, -0.5)
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels([f'col {i}' for i in range(5)], color='#888888', fontsize=9)
ax.set_yticklabels([f'row {i}' for i in range(5)], color='#888888', fontsize=9)
ax.set_title('GridWorld 5×5 — Full Layout', color='white', fontsize=13, pad=12)

# Legend
legend_items = [
    plt.Rectangle((0,0),1,1, fc=cell_colors['start'], ec='#4A9ECC', lw=2),
    plt.Rectangle((0,0),1,1, fc=cell_colors['goal'],  ec=GOLD,      lw=2),
    plt.Rectangle((0,0),1,1, fc=cell_colors['wall'],  ec='#CC4444', lw=2),
    plt.Rectangle((0,0),1,1, fc=cell_colors['empty'], ec='#555555', lw=1),
]
ax.legend(legend_items, ['START (4,1)', 'GOAL (3,4)  reward=+1',
                          'WALL (blocked)', 'Empty  reward=−0.01'],
          loc='lower right', fontsize=9, facecolor='#1A1A1A',
          edgecolor='#555555', labelcolor='white')

plt.tight_layout()
plt.show()

print("Actions available from any non-wall cell:")
for a, (dr, dc) in GridWorld.ACTIONS.items():
    print(f"  {GridWorld.ACTION_NAME[a]}  action {a}  →  move (Δrow={dr:+d}, Δcol={dc:+d})")
print()
print("If the action would hit a wall or the boundary → agent stays in current cell.")


### Step 2 — One Episode: Random Agent (before training)

Before any learning, the agent picks actions completely at random (ε = 1.0).
Let us trace every single step it takes.


In [ ]:

def trace_episode(env, agent=None, max_steps=60, random=False, seed=0):
    # Run one episode and record the full trajectory.
    # If agent is None or random=True, use random actions.
    # Returns list of (state, action, reward, next_state, done).
    rng_t = np.random.default_rng(seed)
    state = env.reset()
    trajectory = []
    for step in range(max_steps):
        if random or agent is None:
            action = rng_t.integers(4)
        else:
            action = int(np.argmax(agent.Q[state[0], state[1]]))  # greedy
        next_state, reward, done = env.step(action)
        trajectory.append((state, action, reward, next_state, done))
        state = next_state
        if done:
            break
    return trajectory


def show_trajectory(trajectory, title='Episode trajectory', ax=None):
    # Overlay a trajectory on the grid as a coloured path.
    show = ax is None
    env_t = GridWorld()
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    env_t.render(ax=ax)

    # Draw path
    states = [t[0] for t in trajectory] + [trajectory[-1][3]]
    xs = [s[1] for s in states]
    ys = [s[0] for s in states]

    # Colour the path by step number
    n = len(xs)
    cmap = plt.cm.plasma
    for i in range(n - 1):
        frac = i / max(n - 2, 1)
        c    = cmap(frac)
        ax.annotate('', xy=(xs[i+1], ys[i+1]), xytext=(xs[i], ys[i]),
                    arrowprops=dict(arrowstyle='->', color=c, lw=2.5))

    # Mark start with a circle
    ax.plot(xs[0], ys[0], 'o', ms=12, color=BLUE,
            markeredgecolor='white', markeredgewidth=1.5, zorder=5)

    total_r = sum(t[2] for t in trajectory)
    n_steps = len(trajectory)
    reached = trajectory[-1][4]  # done flag

    ax.set_title(f'{title}\n{n_steps} steps | total reward = {total_r:.3f} | '
                 f'{"REACHED GOAL ✓" if reached else "DID NOT REACH GOAL ✗"}',
                 color='white', fontsize=10, pad=8)

    if show:
        plt.tight_layout()
        plt.show()

    return n_steps, total_r, reached


# Random episode
env_demo = GridWorld()
traj_random = trace_episode(env_demo, random=True, max_steps=80, seed=7)

fig, ax = plt.subplots(figsize=(6, 6))
steps, reward, reached = show_trajectory(traj_random, title='Random agent (ε=1)', ax=ax)
plt.tight_layout()
plt.show()

print("Step-by-step log:")
print(f"{'Step':>4}  {'State':>8}  {'Action':>8}  {'Reward':>8}  {'Next state':>12}")
print("-" * 50)
for i, (s, a, r, ns, done) in enumerate(traj_random[:25]):   # show first 25 steps
    print(f"{i+1:>4}  {str(s):>8}  {GridWorld.ACTION_NAME[a]:>8}  {r:>8.3f}  {str(ns):>12}")
if len(traj_random) > 25:
    print(f"  ... ({len(traj_random) - 25} more steps)")


### Step 3 — One Episode: Trained Agent (greedy policy)

Now the same starting state, but the agent uses the learned Q-table and always picks
$\arg\max_a Q(s,a)$. No randomness.


In [ ]:

# Trained agent — greedy episode
env_demo = GridWorld()
traj_trained = trace_episode(env_demo, agent=agent, random=False, max_steps=60, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

show_trajectory(traj_random,  title='Random agent (untrained)', ax=axes[0])
show_trajectory(traj_trained, title='Trained agent  (greedy π*)', ax=axes[1])

plt.suptitle('Before vs After Training', color='white', fontsize=13)
plt.tight_layout()
plt.show()

print("\nTrained agent step-by-step:")
print(f"{'Step':>4}  {'State':>8}  {'Action':>8}  {'Reward':>8}  {'Next state':>12}  {'Done':>6}")
print("-" * 60)
for i, (s, a, r, ns, done) in enumerate(traj_trained):
    print(f"{i+1:>4}  {str(s):>8}  {GridWorld.ACTION_NAME[a]:>8}  {r:>8.3f}  {str(ns):>12}  {str(done):>6}")

total_r = sum(t[2] for t in traj_trained)
print(f"\nTotal steps : {len(traj_trained)}")
print(f"Total reward: {total_r:.3f}   (= {len(traj_trained)-1} × (−0.01) + 1.0)")
print(f"\nThe agent found the {'OPTIMAL' if len(traj_trained) <= 8 else 'A'} path.")


### Step 4 — Reading the Q-Table Cell by Cell

The Q-table has **100 numbers** (25 cells × 4 actions). Let us make them readable.

Each cell shows **four Q-values** — one for each action. The agent always picks the
action with the **highest Q-value** (the ↑↓←→ arrow on the policy map).

The Q-value $Q^*(s,a)$ means: *"if I am in state $s$ and take action $a$, then follow
the optimal policy forever, I expect to accumulate this much reward in total."*


In [ ]:

# ── Per-cell Q-value mini-bars ────────────────────────────────────────────────
fig, axes = plt.subplots(5, 5, figsize=(14, 14))
fig.patch.set_facecolor('#1A1A1A')

action_labels = ['↑', '↓', '←', '→']
bar_colors    = [BLUE, BLUE, BLUE, BLUE]

q_min = agent.Q[~np.isinf(agent.Q)].min()
q_max = agent.Q[~np.isinf(agent.Q)].max()

for r in range(5):
    for c in range(5):
        ax = axes[r][c]
        ax.set_facecolor('#2D2D2D')
        for spine in ax.spines.values():
            spine.set_edgecolor('#555555')

        if (r, c) in env.walls:
            ax.set_facecolor('#3D1A1A')
            ax.text(0.5, 0.5, 'WALL', transform=ax.transAxes,
                    ha='center', va='center', color='#CC4444',
                    fontsize=12, fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f'({r},{c})', color='#CC4444', fontsize=8, pad=2)
            continue

        if (r, c) == env.goal:
            ax.set_facecolor('#1A3320')
            ax.text(0.5, 0.6, 'GOAL', transform=ax.transAxes,
                    ha='center', va='center', color=GOLD,
                    fontsize=12, fontweight='bold')
            ax.text(0.5, 0.3, 'reward = +1', transform=ax.transAxes,
                    ha='center', va='center', color='#AAAAAA', fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f'({r},{c})', color=GOLD, fontsize=8, pad=2)
            continue

        q_vals  = agent.Q[r, c]
        best_a  = int(np.argmax(q_vals))
        colors  = ['#CC4444' if i == best_a else '#4A9ECC' for i in range(4)]

        bars = ax.bar(range(4), q_vals, color=colors, width=0.7, alpha=0.85,
                      edgecolor='#222222', linewidth=0.5)

        # Value labels on bars
        for i, (bar, val) in enumerate(zip(bars, q_vals)):
            ypos = val + 0.005 if val >= 0 else val - 0.025
            ax.text(i, ypos, f'{val:.3f}', ha='center', va='bottom',
                    fontsize=5.5, color='white')

        ax.set_xticks(range(4))
        ax.set_xticklabels(action_labels, fontsize=9, color='#AAAAAA')
        ax.set_yticks([])
        ax.set_ylim(q_min - 0.05, q_max + 0.05)
        ax.axhline(0, color='#555555', linewidth=0.5)

        # Cell label + best action
        is_start = (r, c) == env.start
        title_col = BLUE if is_start else 'white'
        ax.set_title(f'({r},{c})  best: {action_labels[best_a]}',
                     color=title_col, fontsize=8, pad=2)

        # Highlight frame for best action
        bars[best_a].set_edgecolor('#E8B931')
        bars[best_a].set_linewidth(2)

fig.suptitle('Q-Table: Every Cell, Every Action\n'
             '(Red/gold bar = best action for that cell)',
             color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### Step 5 — Why Do the Q-Values Have Those Numbers?

The Q-values are **not arbitrary** — they are determined by the Bellman equation.
Let us trace the reasoning for a few key cells.

**Start at the goal cell (3,4):**
The episode ends with reward $+1$. So all Q-values that land in $(3,4)$ get an update
toward $+1$. These cells are directly adjacent to the goal.

**One step back from goal — cell (3,3) is a wall**, so the agent cannot enter from there.
The path to the goal passes through cell **(2,4)** and **(3,3)** is blocked, so the
agent must approach from above: **(2,4)** → right → **(3,4)**.

**Q-value arithmetic (approximate, $\gamma=0.9$):**

| State | Best action | Q-value | Why |
|-------|-------------|---------|-----|
| (3,4) | — | (goal, terminal) | — |
| (2,4) | → Down | ≈ $-0.01 + 0.9 \times 1.0 = 0.89$ | one step from goal |
| (1,4) | → Down | ≈ $-0.01 + 0.9 \times 0.89 = 0.79$ | two steps from goal |
| (0,4) | → Down | ≈ $-0.01 + 0.9 \times 0.79 = 0.70$ | three steps from goal |

The value function is a **gradient field pointing toward the goal** — every cell's
value is slightly less than the value of the next cell on the optimal path.


In [ ]:

# ── Show the value gradient numerically ──────────────────────────────────────
V = np.max(agent.Q, axis=2)   # V(s) = max_a Q(s,a)

print("Value function V*(s) = max_a Q*(s,a)")
print("(higher = closer to goal along optimal path)\n")

print("     col0   col1   col2   col3   col4")
print("     " + "-"*40)
for r in range(5):
    row_str = f"row{r} |"
    for c in range(5):
        if (r,c) in env.walls:
            row_str += "  WALL "
        elif (r,c) == env.goal:
            row_str += "  GOAL "
        else:
            row_str += f" {V[r,c]:6.3f}"
    print(row_str)

print()
print("Bellman arithmetic check — path from start (4,1) to goal (3,4):")
print()

# Trace the optimal path manually
env_t   = GridWorld()
state   = env_t.reset()
gamma   = 0.9
step_n  = 0
print(f"{'Step':>4}  {'State':>8}  {'Action':>8}  {'V*(s)':>8}  "
      f"{'V*(s\')':>8}  {'Expected Q':>12}  {'Actual Q':>10}")
print("-" * 70)

for _ in range(20):
    a      = int(np.argmax(agent.Q[state[0], state[1]]))
    ns, r, done = env_t.step(a)
    q_pred = r + gamma * V[ns[0], ns[1]] * (1 - done)
    q_act  = agent.Q[state[0], state[1], a]
    print(f"{step_n+1:>4}  {str(state):>8}  "
          f"{GridWorld.ACTION_NAME[a]:>8}  "
          f"{V[state[0],state[1]]:>8.4f}  "
          f"{V[ns[0],ns[1]]:>8.4f}  "
          f"{q_pred:>12.4f}  "
          f"{q_act:>10.4f}")
    state  = ns
    step_n += 1
    if done:
        print(f"  → GOAL reached at step {step_n}!")
        break


### Step 6 — The Value Landscape as a Physics Analogy

The value function $V^*(s)$ defines a **landscape** over the grid. Think of it as a
hill — the goal is the peak, and cells further away are lower. The optimal policy
is simply: **always go uphill**.

This is exactly the analogy with a physical particle on a potential surface:
the particle follows the gradient $\nabla V$. Here the "gradient" is discrete —
the agent steps to the neighbouring cell with the highest V value.


In [ ]:

# ── 3D Value landscape ────────────────────────────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D

V      = np.max(agent.Q, axis=2)
V_plot = V.copy()
# Fill walls with NaN for clean plotting
for (r,c) in env.walls:
    V_plot[r, c] = np.nan

fig = plt.figure(figsize=(14, 5))

# ── Left: 3D surface ─────────────────────────────────────────────────────────
ax3d = fig.add_subplot(131, projection='3d')
ax3d.set_facecolor('#1A1A1A')
fig.patch.set_facecolor('#1A1A1A')

rows_g, cols_g = np.mgrid[0:5, 0:5]
surf = ax3d.plot_surface(cols_g, rows_g, V_plot, cmap='plasma',
                         edgecolor='none', alpha=0.9)
ax3d.set_xlabel('col', color='#AAAAAA', fontsize=8)
ax3d.set_ylabel('row', color='#AAAAAA', fontsize=8)
ax3d.set_zlabel('V*(s)', color='#AAAAAA', fontsize=8)
ax3d.set_title('Value Landscape\n(go uphill = optimal policy)',
               color='white', fontsize=10)
ax3d.tick_params(colors='#666666', labelsize=7)

# Mark the goal peak
ax3d.scatter([4], [3], [V[3,4]], color=GOLD, s=80, zorder=5)
ax3d.text(4.2, 3, V[3,4]+0.03, 'GOAL', color=GOLD, fontsize=8)

# ── Middle: 2D heatmap with gradient arrows ──────────────────────────────────
ax2d = fig.add_subplot(132)
ax2d.set_facecolor('#1A1A1A')

V_vis = np.where(
    [[((r,c) in env.walls) for c in range(5)] for r in range(5)],
    np.nan, V
)
img = ax2d.imshow(V_vis, cmap='plasma', interpolation='nearest',
                  vmin=np.nanmin(V_vis), vmax=np.nanmax(V_vis))
plt.colorbar(img, ax=ax2d, label='V*(s)', fraction=0.04)

# Arrows: policy = gradient direction
for r in range(5):
    for c in range(5):
        if (r,c) in env.walls or (r,c) == env.goal:
            continue
        best_a = int(np.argmax(agent.Q[r, c]))
        dr, dc = GridWorld.ACTIONS[best_a]
        ax2d.annotate('', xy=(c + 0.35*dc, r + 0.35*dr),
                      xytext=(c, r),
                      arrowprops=dict(arrowstyle='->', color='white',
                                      lw=1.8, mutation_scale=14))

# Grid lines
for i in range(6):
    ax2d.axvline(i-0.5, color='#555555', lw=0.8)
    ax2d.axhline(i-0.5, color='#555555', lw=0.8)

for (r,c) in env.walls:
    ax2d.text(c, r, '■', ha='center', va='center', color='#CC4444', fontsize=14)
ax2d.text(env.goal[1], env.goal[0], 'G', ha='center', va='center',
          color=GOLD, fontsize=14, fontweight='bold')
ax2d.set_title('Policy = Gradient of V*\n(arrows point uphill)',
               color='white', fontsize=10)
ax2d.set_xticks(range(5)); ax2d.set_yticks(range(5))

# ── Right: optimal path highlighted ─────────────────────────────────────────
ax_path = fig.add_subplot(133)
ax_path.set_facecolor('#1A1A1A')
env_t = GridWorld()
env_t.render(Q=agent.Q, ax=ax_path)

# Trace optimal path and overlay it
env_t2  = GridWorld()
state   = env_t2.reset()
path_states = [state]
for _ in range(30):
    a     = int(np.argmax(agent.Q[state[0], state[1]]))
    ns, r, done = env_t2.step(a)
    path_states.append(ns)
    state = ns
    if done: break

px = [s[1] for s in path_states]
py = [s[0] for s in path_states]
ax_path.plot(px, py, 'o-', color=GOLD, lw=3, ms=10,
             markeredgecolor='white', markeredgewidth=1.5, zorder=6,
             label=f'Optimal path ({len(path_states)-1} steps)')
for i, (x, y) in enumerate(zip(px, py)):
    ax_path.text(x + 0.05, y - 0.3, str(i), color='white', fontsize=8, zorder=7)

ax_path.legend(loc='upper right', fontsize=8, facecolor='#1A1A1A',
               edgecolor='#555555', labelcolor='white')
ax_path.set_title('Optimal Path: START → GOAL\n(numbered steps)',
                  color='white', fontsize=10)

plt.suptitle('The Value Landscape — Connecting Physics and RL',
             color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nOptimal path: {' → '.join(str(s) for s in path_states)}")
print(f"Total steps: {len(path_states)-1}")
print(f"Total reward: {-0.01*(len(path_states)-2) + 1.0:.3f}")


### Step 7 — Watching the Policy Evolve During Training

The agent starts knowing nothing. Let us take snapshots of the policy at
**5 different moments** during training and watch it improve.

At the start: arrows point randomly (whatever action had the first lucky update).
After a few hundred episodes: a rough path begins to appear near the goal.
After 2000 episodes: the policy is stable and optimal.


In [ ]:

# ── Policy snapshots during training ─────────────────────────────────────────
snapshot_episodes = [10, 50, 200, 500, 2000]
snapshot_agents   = []

env_snap = GridWorld()
ag_snap  = QLearningAgent(alpha=0.1, gamma=0.9, tau=300, eps_max=1.0, eps_min=0.01,
                          seed=SEED)

ep = 0
snap_idx = 0
while snap_idx < len(snapshot_episodes):
    ag_snap.run_episode(env_snap)
    ep += 1
    if ep == snapshot_episodes[snap_idx]:
        snapshot_agents.append((ep, copy.deepcopy(ag_snap.Q)))
        snap_idx += 1

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.patch.set_facecolor('#1A1A1A')

for ax, (ep_n, Q_snap) in zip(axes, snapshot_agents):
    env_snap.render(Q=Q_snap, ax=ax)
    # Compute greedy path
    env_t3 = GridWorld()
    s      = env_t3.reset()
    path   = [s]
    visited = {s}
    for _ in range(40):
        a  = int(np.argmax(Q_snap[s[0], s[1]]))
        ns, _, done = env_t3.step(a)
        if ns in visited and not done:
            break   # stuck in a loop
        path.append(ns)
        visited.add(ns)
        s = ns
        if done:
            break

    px = [p[1] for p in path]
    py = [p[0] for p in path]
    ax.plot(px, py, 'o-', color=GOLD, lw=2.5, ms=7,
            markeredgecolor='white', markeredgewidth=1, zorder=6, alpha=0.9)

    # Did it reach goal?
    reached = path[-1] == env.goal
    status  = f"✓ {len(path)-1} steps" if reached else "✗ lost"
    ax.set_title(f'After {ep_n} episodes\n{status}',
                 color=(GOLD if reached else RED), fontsize=10, pad=6)

plt.suptitle('Policy Evolution During Training  (gold line = greedy path from start)',
             color='white', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()
print("As training progresses:")
print("  • Early: agent wanders randomly, Q-values near zero everywhere")
print("  • Mid:   agent finds the goal occasionally, value signal propagates backward")
print("  • Late:  every cell has a clear best action — the policy is stable")


### Step 8 — The Q-Value Propagation: How Values Flow Backward

The most important thing to understand about Q-learning is **how information travels
from the goal back to the start**. It does not happen in one episode — it takes
many episodes for the $+1$ reward signal to propagate across the grid.

Think of it like a diffusion process: the goal is a source of "value", and each
Bellman update spreads that value one step further back.


In [ ]:

# ── Show how max Q-value propagates outward from goal ─────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.patch.set_facecolor('#1A1A1A')

checkpoints = [1, 5, 20, 100, 300, 500, 800, 1200, 1700, 2000]
env_prop    = GridWorld()
ag_prop     = QLearningAgent(alpha=0.1, gamma=0.9, tau=400, eps_max=1.0, eps_min=0.01,
                             seed=SEED+1)

checkpoint_Qs = {}
ep = 0
for target in checkpoints:
    while ep < target:
        ag_prop.run_episode(env_prop)
        ep += 1
    checkpoint_Qs[target] = ag_prop.Q.copy()

vmax_global = np.max([np.max(Q) for Q in checkpoint_Qs.values()])
vmin_global = 0.0   # start from 0

for idx, (target, ax) in enumerate(zip(checkpoints, axes.flat)):
    Q_cp = checkpoint_Qs[target]
    V_cp = np.max(Q_cp, axis=2)

    # Mask walls
    V_masked = np.where(
        [[((r,c) in env.walls) for c in range(5)] for r in range(5)],
        np.nan, V_cp
    )

    img = ax.imshow(V_masked, cmap='plasma', interpolation='nearest',
                    vmin=vmin_global, vmax=vmax_global)

    for r in range(5):
        for c in range(5):
            if (r,c) in env.walls:
                ax.text(c, r, '■', ha='center', va='center', color='#CC4444', fontsize=12)
            elif (r,c) == env.goal:
                ax.text(c, r, 'G', ha='center', va='center', color=GOLD,
                        fontsize=13, fontweight='bold')
            else:
                v = V_cp[r,c]
                ax.text(c, r, f'{v:.2f}', ha='center', va='center',
                        color='white', fontsize=7.5)

    for i in range(6):
        ax.axvline(i-0.5, color='#333333', lw=0.6)
        ax.axhline(i-0.5, color='#333333', lw=0.6)

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Episode {target}', color='white', fontsize=9, pad=4)
    ax.set_facecolor('#1A1A1A')

plt.suptitle('Value Propagation: How V*(s) Spreads Backward from Goal (plasma = high value)',
             color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("Notice:")
print("  • Episode 1-5:   only cells adjacent to goal have non-zero values")
print("  • Episode 20-100: values propagate one more step per ~10 episodes")
print("  • Episode 300+:   the gradient field is nearly complete")
print("  • Episode 2000:   all reachable cells have stable, accurate values")


### Summary: What the Final Answer Means

After training, we have found the **optimal policy** $\pi^*(s)$ for GridWorld.
Here is what that means concretely:

| Question | Answer |
|----------|--------|
| What is the optimal policy? | A rule: in each cell, take the action with the highest Q-value |
| What path does it produce? | The shortest path from (4,1) to (3,4) that avoids walls |
| What is the optimal value at the start? | $V^*(4,1) = 1.0 - n_{\text{steps}} \times 0.01$ |
| How many steps is optimal? | Determined by the maze geometry |
| What do the Q-values represent? | Expected total future reward (discounted) from that (state, action) |
| Why do values decrease away from goal? | Because $\gamma = 0.9 < 1$ discounts future rewards |
| How is this related to physics? | V* is the "potential field"; policy = gradient ascent on V* |

The trained Q-table is **the complete solution to the GridWorld problem** — for every
possible position the agent could find itself in, it knows the best possible action.


In [ ]:

# Final clean summary figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#1A1A1A')

V_final = np.max(agent.Q, axis=2)

# ── Panel 1: Policy ──────────────────────────────────────────────────────────
env.render(Q=agent.Q, ax=axes[0])
# Trace optimal path
env_f = GridWorld()
s     = env_f.reset()
path  = [s]
for _ in range(30):
    a     = int(np.argmax(agent.Q[s[0], s[1]]))
    ns, _, done = env_f.step(a)
    path.append(ns)
    s = ns
    if done: break
px = [p[1] for p in path]; py = [p[0] for p in path]
axes[0].plot(px, py, 'o-', color=GOLD, lw=3, ms=10,
             markeredgecolor='white', markeredgewidth=1.5, zorder=6)
for i, (x, y) in enumerate(zip(px, py)):
    axes[0].text(x+0.05, y-0.28, str(i), color='white', fontsize=9, fontweight='bold', zorder=7)
axes[0].set_title(f'Optimal Policy π*(s)\nPath: {len(path)-1} steps to goal',
                  color='white', fontsize=11, pad=8)

# ── Panel 2: Value heatmap ───────────────────────────────────────────────────
V_disp = np.where([[((r,c) in env.walls) for c in range(5)] for r in range(5)],
                  np.nan, V_final)
img2 = axes[1].imshow(V_disp, cmap='plasma', interpolation='nearest')
plt.colorbar(img2, ax=axes[1], label='V*(s)', fraction=0.04)
for r in range(5):
    for c in range(5):
        if (r,c) in env.walls:
            axes[1].text(c, r, '■', ha='center', va='center', color='#CC4444', fontsize=14)
        elif (r,c) == env.goal:
            axes[1].text(c, r, 'G', ha='center', va='center', color=GOLD,
                         fontsize=14, fontweight='bold')
        else:
            axes[1].text(c, r, f'{V_final[r,c]:.3f}', ha='center', va='center',
                         color='white', fontsize=10)
for i in range(6):
    axes[1].axvline(i-0.5, color='#555555', lw=0.8)
    axes[1].axhline(i-0.5, color='#555555', lw=0.8)
axes[1].set_xticks(range(5)); axes[1].set_yticks(range(5))
axes[1].set_title('Value Function V*(s)\n(brightness = how good it is to be here)',
                  color='white', fontsize=11, pad=8)

# ── Panel 3: Q-value bar for start state ─────────────────────────────────────
ax3 = axes[2]
q_start = agent.Q[env.start[0], env.start[1]]
best_a  = int(np.argmax(q_start))
bar_cols = [GOLD if i == best_a else BLUE for i in range(4)]
bars = ax3.bar(['↑ Up', '↓ Down', '← Left', '→ Right'], q_start,
               color=bar_cols, edgecolor='#222222', linewidth=1, alpha=0.9)
for bar, val in zip(bars, q_start):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.005,
             f'{val:.4f}', ha='center', va='bottom', color='white', fontsize=10)
ax3.set_title(f'Q-values at START cell {env.start}\n'
              f'Best action: {GridWorld.ACTION_NAME[best_a]} (gold bar)',
              color='white', fontsize=11, pad=8)
ax3.set_ylabel('Q*(start, action)', color='#AAAAAA')
ax3.grid(True, axis='y', alpha=0.4)
ax3.axhline(0, color='#555555', lw=0.8)

plt.suptitle('The Complete Solution to GridWorld', color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The optimal policy in plain language:")
action_names_full = {0:'Go UP   ', 1:'Go DOWN ', 2:'Go LEFT ', 3:'Go RIGHT'}
for r in range(5):
    for c in range(5):
        if (r,c) in env.walls:
            print(f"  Cell ({r},{c}): WALL — unreachable")
        elif (r,c) == env.goal:
            print(f"  Cell ({r},{c}): GOAL — episode ends here")
        else:
            best = int(np.argmax(agent.Q[r,c]))
            tag  = " ← you are here (start)" if (r,c)==env.start else ""
            print(f"  Cell ({r},{c}): {action_names_full[best]}{tag}")


---
## 10. Verifying the Bellman Equation Numerically

After training, $Q^*$ should satisfy the Bellman optimality equation:

$$Q^*(s,a) \approx R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, \max_{a'} Q^*(s',a')$$

In our deterministic GridWorld, $P(s'|s,a) = 1$ for the resulting $s'$, so:

$$Q^*(s,a) \approx R(s,a) + \gamma \max_{a'} Q^*(s',a')$$

Let's check this for every non-wall, non-goal state-action pair.


In [ ]:
# Verify Bellman consistency of the learned Q-table
print("Bellman Consistency Check: Q*(s,a) vs R + γ·max_a' Q*(s',a')")
print(f"{'State':>8} {'Action':>8} {'Q*(s,a)':>10} {'Bellman RHS':>12} {'|Error|':>10}")
print("-" * 56)

errors = []
for r in range(5):
    for c in range(5):
        if (r,c) in env.walls or (r,c) == env.goal:
            continue
        for a in range(4):
            # Simulate one step
            env.state = (r, c)
            next_state, reward, done = env.step(a)
            env.state = (r, c)   # reset

            q_sa = agent.Q[r, c, a]

            if done:
                bellman_rhs = reward
            else:
                nr, nc = next_state
                bellman_rhs = reward + 0.9 * np.max(agent.Q[nr, nc])

            err = abs(q_sa - bellman_rhs)
            errors.append(err)

            if err > 0.01:  # only print cells with notable residuals
                print(f"  ({r},{c})   {GridWorld.ACTION_NAME[a]:>6}    "
                      f"{q_sa:>10.4f}   {bellman_rhs:>12.4f}   {err:>10.5f}")

print()
print(f"Max  Bellman residual: {max(errors):.5f}")
print(f"Mean Bellman residual: {np.mean(errors):.5f}")
print(f"RMS  Bellman residual: {np.sqrt(np.mean(np.array(errors)**2)):.5f}")
print()
if max(errors) < 0.05:
    print("✓ Q-table satisfies Bellman equation to within 5% — training converged.")
else:
    print("⚠ Residuals are non-negligible — consider more training episodes.")


---
## 11. Hyperparameter Experiments

### Effect of learning rate α and cooling timescale τ

These experiments let you see the contraction mapping in action: too large an α introduces
instability; too small makes learning slow. Too fast annealing (small τ) traps the agent
in suboptimal policies; too slow is wasteful.


In [ ]:
# Experiment 1: Effect of learning rate alpha
alphas   = [0.01, 0.1, 0.5, 0.9]
n_ep     = 1000
env_     = GridWorld()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors_exp = [BLUE, GOLD, RED, '#AA44AA']

for alpha, col in zip(alphas, colors_exp):
    ag = QLearningAgent(alpha=alpha, gamma=0.9, tau=300, seed=SEED)
    for _ in range(n_ep): ag.run_episode(env_)
    sm = smooth(ag.reward_history, 40)
    axes[0].plot(np.arange(len(sm)), sm, color=col, label=f'α={alpha}', linewidth=2)

axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Return (smoothed)')
axes[0].set_title('Effect of Learning Rate α', color='white')
axes[0].legend()
axes[0].grid(True)

# Experiment 2: Effect of annealing timescale tau
taus_exp = [50, 150, 300, 600]
for tau, col in zip(taus_exp, colors_exp):
    ag = QLearningAgent(alpha=0.1, gamma=0.9, tau=tau, seed=SEED)
    for _ in range(n_ep): ag.run_episode(env_)
    sm = smooth(ag.reward_history, 40)
    axes[1].plot(np.arange(len(sm)), sm, color=col, label=f'τ={tau}', linewidth=2)

axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Return (smoothed)')
axes[1].set_title('Effect of Annealing Timescale τ', color='white')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Hyperparameter Sensitivity', color='white', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Experiment 3: Optimistic initialisation vs zero initialisation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
n_ep = 1000

configs = [
    {'init_value': 0.0,  'eps_max': 1.0,  'label': 'Zero init + ε-greedy', 'color': BLUE},
    {'init_value': 2.0,  'eps_max': 0.0,  'label': 'Optimistic init (greedy only)', 'color': GOLD},
    {'init_value': 0.0,  'eps_max': 0.0,  'label': 'Zero init + greedy (no explore)', 'color': RED},
]

for cfg in configs:
    ag = QLearningAgent(alpha=0.1, gamma=0.9, tau=300,
                        init_value=cfg['init_value'],
                        eps_max=cfg['eps_max'], eps_min=0.0, seed=SEED)
    for _ in range(n_ep): ag.run_episode(env_)

    sm_r = smooth(ag.reward_history, 40)
    sm_s = smooth(ag.steps_history, 40)
    axes[0].plot(np.arange(len(sm_r)), sm_r, color=cfg['color'],
                 label=cfg['label'], linewidth=2)
    axes[1].plot(np.arange(len(sm_s)), sm_s, color=cfg['color'],
                 label=cfg['label'], linewidth=2)

for ax, title in zip(axes, ['Return', 'Steps to goal']):
    ax.set_xlabel('Episode')
    ax.set_ylabel(title + ' (smoothed)')
    ax.set_title(f'{title} — Exploration strategies', color='white')
    ax.legend(fontsize=9)
    ax.grid(True)

plt.suptitle('Optimistic Initialisation vs ε-Greedy', color='white', fontsize=13)
plt.tight_layout()
plt.show()
print("Optimistic initialisation achieves systematic exploration without an explicit ε parameter.")
print("The agent is always 'disappointed' by reality and keeps exploring until Q-values stabilise.")


---
## 12. Convergence — The Contraction Mapping in Action

We train three agents from **different random initialisations** and verify they all converge
to the same $Q^*$. This is the Banach fixed-point theorem made visible.


In [ ]:
# Train three agents from very different initialisations
init_values = [0.0, 5.0, -5.0]
labels      = ['Q₀ = 0 (standard)', 'Q₀ = +5 (optimistic)', 'Q₀ = −5 (pessimistic)']
colors_c    = [BLUE, GOLD, RED]
n_ep        = 3000
env_        = GridWorld()

agents = []
for iv in init_values:
    ag = QLearningAgent(alpha=0.1, gamma=0.9, tau=500,
                        eps_max=1.0, eps_min=0.01, init_value=iv, seed=SEED)
    for _ in range(n_ep): ag.run_episode(env_)
    agents.append(ag)

# Compare final Q-tables
Q_ref = agents[0].Q.copy()
print("Max difference between final Q-tables (should be small):")
for i, (ag, label) in enumerate(zip(agents[1:], labels[1:])):
    diff = np.max(np.abs(ag.Q - Q_ref))
    print(f"  |Q({label}) - Q(Q₀=0)|∞ = {diff:.5f}")

# Plot convergence of Q-value for one representative state-action pair
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# We'll track a specific Q(4,1,↑) which is the start-state going Up
state_r, state_c, action = 4, 1, 0
label_sa = f"Q(start=(4,1), ↑)"

for ag, label, col in zip(agents, labels, colors_c):
    # Re-run and record Q(4,1,0) after each episode
    env_t = GridWorld()
    ag_t  = QLearningAgent(alpha=0.1, gamma=0.9, tau=500,
                           eps_max=1.0, eps_min=0.01,
                           init_value={'Q₀ = 0 (standard)': 0.0,
                                       'Q₀ = +5 (optimistic)': 5.0,
                                       'Q₀ = −5 (pessimistic)': -5.0}[label],
                           seed=SEED)
    q_trace = []
    for _ in range(n_ep):
        ag_t.run_episode(env_t)
        q_trace.append(ag_t.Q[state_r, state_c, action])

    axes[0].plot(q_trace, color=col, label=label, linewidth=1.5, alpha=0.9)

axes[0].axhline(Q_ref[state_r, state_c, action], color='white',
                linestyle=':', linewidth=1.5, label='Converged value')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel(f'{label_sa}')
axes[0].set_title('Convergence from Different Initialisations', color='white')
axes[0].legend(fontsize=9)
axes[0].grid(True)

# Right: pairwise Q-table differences over training (approximate)
sm_rd = smooth(agents[0].reward_history, 50)
sm_r1 = smooth(agents[1].reward_history, 50)
sm_r2 = smooth(agents[2].reward_history, 50)
n_sm  = min(len(sm_rd), len(sm_r1), len(sm_r2))

for arr, label, col in zip([sm_rd, sm_r1, sm_r2], labels, colors_c):
    axes[1].plot(np.arange(n_sm), arr[:n_sm], color=col, label=label, linewidth=1.5)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Return (smoothed)')
axes[1].set_title('Return curves all converge', color='white')
axes[1].legend(fontsize=9)
axes[1].grid(True)

plt.suptitle('Banach Fixed-Point Theorem — Q* is unique regardless of initialisation',
             color='white', fontsize=12)
plt.tight_layout()
plt.show()


---
## 13. Guided Exercises

### Exercise A — Stochastic GridWorld ⭐⭐

Modify the `GridWorld.step()` method to add stochasticity: with probability `p_slip = 0.1`,
the agent moves in a random direction instead of the chosen one (slippery floor).

1. What happens to the learned Q-values compared to the deterministic case?
2. Does the agent still find the optimal path? Why or why not?
3. How does the required number of training episodes change?

```python
# Starter code
class StochasticGridWorld(GridWorld):
    def __init__(self, p_slip=0.1):
        super().__init__()
        self.p_slip = p_slip

    def step(self, action):
        if rng.random() < self.p_slip:
            action = rng.integers(4)   # random slip
        return super().step(action)
```


In [ ]:
# YOUR CODE HERE — Implement StochasticGridWorld and train an agent on it.
# Compare the learned policy and value function with the deterministic case.

class StochasticGridWorld(GridWorld):
    def __init__(self, p_slip=0.1):
        super().__init__()
        self.p_slip = p_slip

    def step(self, action):
        if rng.random() < self.p_slip:
            action = rng.integers(4)
        return super().step(action)

# Train and compare
env_det  = GridWorld()
env_sto  = StochasticGridWorld(p_slip=0.15)
n_ep     = 3000

ag_det = QLearningAgent(alpha=0.1, gamma=0.9, tau=400, seed=SEED)
ag_sto = QLearningAgent(alpha=0.1, gamma=0.9, tau=400, seed=SEED)

for _ in range(n_ep):
    ag_det.run_episode(env_det)
    ag_sto.run_episode(env_sto)

# Plot both learned policies side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
env_det.render(Q=ag_det.Q, ax=axes[0]); axes[0].set_title('Deterministic (p_slip=0)', color='white')
env_sto.render(Q=ag_sto.Q, ax=axes[1]); axes[1].set_title('Stochastic (p_slip=0.15)', color='white')
plt.tight_layout()
plt.show()


### Exercise B — SARSA vs Q-Learning ⭐⭐⭐

Q-learning is an **off-policy** algorithm: it always updates toward the *best* next action,
regardless of what the agent will actually do. **SARSA** is **on-policy**: it updates toward
the action the agent *will actually take* under the current policy.

$$\text{SARSA: } Q(s,a) \leftarrow Q(s,a) + \alpha\bigl[r + \gamma\, Q(s', a') - Q(s,a)\bigr]$$

where $a'$ is the action actually selected for the next step, not $\max_{a'}$.

1. Implement a SARSA agent by modifying `QLearningAgent`.
2. Train both on GridWorld and compare convergence speed and final policy.
3. In which cases would SARSA be preferable to Q-learning?


In [ ]:
# YOUR CODE HERE — Implement SARSA

class SARSAAgent(QLearningAgent):
    """
    SARSA (on-policy TD control).
    Key difference from Q-learning: uses the actual next action a' in the update,
    not max_a' Q(s', a').
    """

    def run_episode(self, env, max_steps=200):
        state  = env.reset()
        action = self.select_action(state)   # select first action before loop
        total_reward = 0.0
        td_errors    = []

        for step in range(max_steps):
            next_state, r, done = env.step(action)

            if done:
                td_target = r
            else:
                next_action = self.select_action(next_state)   # on-policy!
                td_target   = r + self.gamma * self.Q[next_state[0], next_state[1], next_action]

            td_error = td_target - self.Q[state[0], state[1], action]
            self.Q[state[0], state[1], action] += self.alpha * td_error

            td_errors.append(abs(td_error))
            total_reward += r

            if done:
                break

            state  = next_state
            action = next_action   # carry over — this is what makes it on-policy

        self.episode += 1
        self.epsilon_history.append(self.epsilon)
        self.reward_history.append(total_reward)
        self.td_error_history.append(np.mean(td_errors))
        self.steps_history.append(step + 1)

        return total_reward, step + 1, np.mean(td_errors)

# Compare SARSA vs Q-learning
env_ = GridWorld()
n_ep = 2000

ag_q    = QLearningAgent(alpha=0.1, gamma=0.9, tau=300, seed=SEED)
ag_sarsa = SARSAAgent(alpha=0.1, gamma=0.9, tau=300, seed=SEED)

for _ in range(n_ep):
    ag_q.run_episode(env_)
    ag_sarsa.run_episode(env_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ag, label, col in [(ag_q, 'Q-learning', BLUE), (ag_sarsa, 'SARSA', GOLD)]:
    sm = smooth(ag.reward_history, 50)
    axes[0].plot(np.arange(len(sm)), sm, color=col, label=label, linewidth=2)
    sm_s = smooth(ag.steps_history, 50)
    axes[1].plot(np.arange(len(sm_s)), sm_s, color=col, label=label, linewidth=2)

for ax, title in zip(axes, ['Return', 'Steps']):
    ax.set_xlabel('Episode'); ax.set_ylabel(title + ' (smoothed)')
    ax.set_title(f'{title} — Q-Learning vs SARSA', color='white')
    ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()


### Exercise C — Larger Grid and Maze ⭐⭐⭐

Extend GridWorld to a 10×10 grid with a more complex maze layout.

1. How does the Q-table size scale with grid dimensions?
2. How many episodes are needed for convergence? Plot the scaling.
3. At what point does the tabular Q-table become impractical? (Motivates next week's DQN.)


In [ ]:
# YOUR CODE HERE — Build a larger GridWorld and study scaling

def make_maze(rows, cols, seed=42):
    """Generate a simple random maze as a set of wall positions."""
    rng_m = np.random.default_rng(seed)
    walls = set()
    # Add random interior walls, avoiding start and goal
    start = (rows-1, 0)
    goal  = (0, cols-1)
    for _ in range(rows * cols // 5):
        r = rng_m.integers(1, rows-1)
        c = rng_m.integers(1, cols-1)
        if (r,c) != start and (r,c) != goal:
            walls.add((r,c))
    return walls

# Study scaling: Q-table size and convergence vs grid size
grid_sizes = [3, 5, 7, 10]
qtable_sizes = [n**2 * 4 for n in grid_sizes]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(grid_sizes, qtable_sizes, color=BLUE, alpha=0.8, width=0.8)
for n, q in zip(grid_sizes, qtable_sizes):
    ax.text(n, q + 20, str(q), ha='center', color='white', fontsize=10)
ax.set_xlabel('Grid size n×n')
ax.set_ylabel('Q-table entries (n² × 4)')
ax.set_title('Q-Table Size Scales as O(n²)', color='white')
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

print("Grid size → Q-table entries:")
for n, q in zip(grid_sizes, qtable_sizes):
    print(f"  {n:>3}×{n:<3} →  {q:>6} entries")
print()
print("At 100×100: Q-table has 40,000 entries — still tractable.")
print("At state spaces like Atari (~10^70 states): tabular RL is impossible.")
print("→ Motivation for Deep Q-Networks (DQN) — Week 6.")


---
## 14. Summary

### What we built

| Concept | Implementation |
|---------|---------------|
| MDP framework | `GridWorld` class — state, action, reward, terminal |
| Value function | Derived from Bellman equation, visualised as heatmap |
| Bellman equation | Verified numerically as fixed-point condition |
| Q-learning | `QLearningAgent` — tabular, ε-greedy, annealing |
| Simulated annealing | ε-annealing schedule, compared with physical SA |
| Convergence | Demonstrated independence from initialisation |
| SARSA | On-policy variant — Exercise B |

### Key equations

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)\bigl[R(s,a,s') + \gamma V^\pi(s')\bigr]$$

$$Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \max_{a'} Q^*(s',a')$$

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{TD target}} - Q(s,a)\bigr]$$

### Looking ahead — Week 6

Next week we move from hand-coded environments to **Gymnasium** (formerly OpenAI Gym).
We will run CartPole and FrozenLake out of the box, train them with **Stable-Baselines3**,
and analyse the learned policies. The Q-learning foundations from this week carry directly
into understanding what Stable-Baselines3 is doing under the hood.

The conceptual thread continues: emergence from optimisation, with the same Bellman equation
at the heart of every modern RL algorithm.

---
*Week 5 — Reinforcement Learning: The Idea*  
*From Conway to LangGraph | University of Bologna | Prof. Degli Esposti*
